In [9]:
%%bash
# extract genomad information for all UHVDB species reps
cat /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/results/uhvdb/mine/*/*/*/genomad/*_virus_summary.tsv.gz \
    > combined_genomad_virus_summary.tsv.gz

In [10]:
import polars as pl

# load all genomad virus sumamries
virus_summary_all = pl.read_csv('combined_genomad_virus_summary.tsv.gz', separator='\t', ignore_errors=True)

In [19]:
# load all UHGV virus summaries
uhgv_metadata = pl.read_csv('https://portal.nersc.gov/cfs/m342/UHGV/metadata/uhgv_metadata.tsv', separator='\t', ignore_errors=True)

In [ ]:
# load UHVDB clustering information
uhvdb_r2025_10 = pl.read_csv('/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_clustering_thru_votus.tsv', separator='\t')

In [15]:
# extract species reps
uhvdb_r2025_10_species_reps = set(uhvdb_r2025_10['votu_rep'])
len(uhvdb_r2025_10_species_reps)

218733

In [25]:
# filter genomad to only UHVDB species reps
virus_summary_filt = virus_summary_all.filter(pl.col('seq_name').is_in(uhvdb_r2025_10_species_reps)).unique('seq_name')
print(len(virus_summary_filt))

173252


In [26]:
# filter UHGV metadata to only UHVDB species reps
uhgv_metadata_filt = uhgv_metadata.filter(pl.col('uhgv_genome').is_in(uhvdb_r2025_10_species_reps))
print(len(uhgv_metadata_filt))

45479


In [31]:
# combine genetic code information for each sequence
combined_gcode_file = pl.concat([
    virus_summary_filt[['seq_name', 'genetic_code']],
    uhgv_metadata_filt.rename({'uhgv_genome': 'seq_name'})[['seq_name', 'genetic_code']]
])

# write out file
combined_gcode_file.write_csv('uhvdb_r2025_10_species_reps_genetic_code.tsv', separator='\t')

In [ ]:
%%bash
# run functional annotation on species reps
cd /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/genomeannotation

# nextflow run . \
#     -c /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb/assets/configs/conf/uw_hyak.config \
#     -w 
#     --input_fna /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/uhvdb_final_files/r2025_10/uhvdb_votu_reps.fna.gz \
#     --input_tsv /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/uhvdb-manuscript/figure_4/uhvdb_r2025_10_species_reps_genetic_code.tsv \
#     --chunk_size 1000 \
#     --bakta_db_version='full' \
#     --output uhvdb_r2025_10

seq_name,genetic_code
str,i64
"""ERZ25233354.58""",11
"""ERZ21615637.505""",11
"""v42996""",11
"""ERZ25878340.43413""",11
"""ERZ989762.40354""",11
…,…
"""UHGV-2239868""",11
"""UHGV-2241160""",11
"""UHGV-2241301""",11
